In [ ]:
import importlib.util, subprocess, sys
for _p in ['polars', 'arch']:
    if importlib.util.find_spec(_p) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _p], check=False)

In [ ]:
import json, base64, io, contextlib, traceback, time
import matplotlib; matplotlib.use("Agg")
P = json.loads(base64.b64decode("...").decode())
report = []
for nb in P:
    base = {}
    try:
        exec(nb["setup"], base)
    except Exception as e:
        report.append(f"{nb['file']}: SETUP FAILED {type(e).__name__}: {e}"); continue
    for e in nb["ex"]:
        t0 = time.time()
        R = dict(base); exec(e["solution"], R)
        U = dict(base); exec(e["solution"], U)
        tns = dict(base); tns["U"], tns["R"] = U, R
        try:
            exec(e["tests"], tns)
            bad = []
            for name, fn in tns["TESTS"]:
                try: fn()
                except Exception as err: bad.append(f"{name}: {type(err).__name__}: {err}")
        except Exception as err:
            bad = ["tests crashed: " + repr(err)]
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                exec(e["demo"], dict(U))
        except Exception as err:
            bad.append(f"demo: {type(err).__name__}: {err}")
        report.append(f"{e['id']:4s} {'OK  ' if not bad else 'FAIL'} {time.time() - t0:5.1f}s " + " | ".join(bad))
print("\n".join(report))
print("SELFTEST DONE", sum(r.split()[1] == "OK" for r in report if len(r.split()) > 1), "ok of", len(report))
